# Problem 2. 두 문장의 의미 유사도 측정하기

## 문제 개요
검색 서비스는 사용자의 질문과 의미가 같은 기존 질문을 찾기 위해 **두 문장의 의미 유사도**를 0~5점으로 매기려 합니다.
문장 쌍의 유사도 점수를 예측하세요.

## 파일 구조
```
Problem2/
├── problem2.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 8,000쌍 (score 포함)
- `data/test.csv` : 3,668쌍 (score 없음)

| 컬럼 | 설명 |
|---|---|
| id | 문장 쌍 고유 번호 (`S00000` 형식) |
| sentence1, sentence2 | 비교할 두 문장 (숙소 후기, 정책 문서, 스마트홈 질문 등) |
| **score** | **의미 유사도 0(전혀 다름) ~ 5(완전히 같음), 예측 대상** |

### 데이터 특징
- 여러 사람이 매긴 점수의 평균이라 소수점 값입니다.
- 단어가 많이 겹쳐도 뜻이 다를 수 있고("A는 되나요?" vs "A는 안 되나요?"), 단어가 달라도 뜻이 같을 수 있습니다(번역투 바꿔 쓰기).
- 출처: KLUE-STS (KLUE Benchmark, CC BY-SA 4.0) 학습 데이터를 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem2/`)에 **`submission.csv`** 로 저장

## 평가
**피어슨 상관계수 (Pearson correlation)** — 높을수록 좋습니다.

- 점수의 절대값보다 **높고 낮음의 경향**이 정답과 얼마나 일치하는지를 봅니다.

## 초기 코드 (베이스라인)
아래 베이스라인은 학습 없이 두 문장의 단어 TF-IDF 코사인 유사도에 5를 곱해 점수로 씁니다. 동작은 하지만 점수가 낮습니다.

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv")
# 두 문장의 단어 TF-IDF 코사인 유사도 × 5 를 그대로 점수로 사용 (학습 없음)
vec = TfidfVectorizer().fit(pd.concat([train.sentence1, train.sentence2]))
a, b = vec.transform(test.sentence1), vec.transform(test.sentence2)
cos = np.asarray(a.multiply(b).sum(1)).ravel()
pd.DataFrame({"id": test.id, "score": cos * 5}).to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| id | score |
|---|---|
| S08000 | 3.2 |
| S08001 | 0.4 |

- 컬럼명 `id`, `score`, 결측 없이 모든 id 포함
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()